# Smart Travel - Machine Learning

This notebook introduces the first Machine Learning model in the Smart Travel project.

The goal is not to apply an algorithm blindly. The goal is to answer a real question:

**Can destinations be grouped naturally based on their characteristics?**

## Method

We use **K-Means clustering**, an unsupervised learning algorithm.

Unsupervised means we do not give the model labels such as `Beach`, `City Break`, or `Nature`. Instead, the model tries to discover groups based on similarities in the data.

Important ideas:

- A **cluster** is a group of similar destinations.
- A **centroid** is the center of a cluster.
- K-Means assigns each destination to the nearest centroid.
- We must choose `k`, the number of clusters.
- We standardize the data so large count variables do not dominate the model.

## Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams["figure.figsize"] = (10, 6)

DATA_PATH = "../data/processed/destinations_feature_engineered.csv"
df = pd.read_csv(DATA_PATH)

## Step 1 - Choose Model Features

### Question

Which columns should the model see?

The model should not see text identifiers such as `destination_name`, `country`, or `search_name`.

For the first model, it should also not see interpreted features such as `dominant_travel_style`, `balanced_destination`, or `warm_destination`.

We want K-Means to discover groups from raw evidence: weather and place counts.

### Analysis

In [ ]:
df[[
    "destination_name",
    "country",
    "dominant_travel_style",
    "climate_category",
    "restaurant_count",
    "cafe_count",
    "bar_count",
    "museum_count",
    "park_count",
    "beach_count",
    "summer_avg_temp",
    "summer_avg_daily_rain",
]].head()

### Insight

For the first clustering model, we use raw POI counts and weather variables instead of user-facing scores. Scores remain useful for interpretation, but the model starts from the underlying data.

## Step 1 Continued - Final Feature Set

### Question

Which exact variables go into K-Means?

### Analysis

In [ ]:
model_features = [
    "restaurant_count",
    "cafe_count",
    "bar_count",
    "museum_count",
    "park_count",
    "beach_count",
    "summer_avg_temp",
    "summer_avg_daily_rain",
]

X = df[model_features]
X.describe().T

### Insight

The selected features mix destination experience and climate. We intentionally keep count variables as counts. The important condition is that they must be standardized before K-Means because their raw scales are very different.

## Step 2 - StandardScaler

### Question

Why do we need to standardize the data before K-Means?

### Analysis

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled_df = pd.DataFrame(X_scaled, columns=model_features)
X_scaled_df.describe().T

### Insight

After standardization, each feature has mean close to 0 and standard deviation close to 1. This prevents variables like `restaurant_count` from dominating variables like `beach_count` or rainfall only because their raw values are larger.

## Step 3 - Elbow Method And Silhouette Score

### Question

How many clusters should we ask K-Means to find?

### Method

We compare several values of `k` using two indicators:

- **Inertia**: lower is better, but it always decreases as k increases. We look for an elbow.
- **Silhouette score**: higher is better. It measures how well-separated the clusters are.

In [ ]:
k_values = range(2, 7)
inertias = []
silhouette_scores = []

for k in k_values:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X_scaled)

    inertias.append(model.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, labels))

cluster_metrics = pd.DataFrame({
    "k": list(k_values),
    "inertia": inertias,
    "silhouette_score": silhouette_scores,
})

cluster_metrics

The same metrics are also saved in `../data/processed/kmeans_cluster_metrics.csv` so the result can be reviewed without rerunning the model.

In [ ]:
saved_cluster_metrics = pd.read_csv("../data/processed/kmeans_cluster_metrics.csv")
saved_cluster_metrics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.lineplot(data=cluster_metrics, x="k", y="inertia", marker="o", ax=axes[0])
axes[0].set_title("Elbow Method")
axes[0].set_xlabel("Number of clusters")
axes[0].set_ylabel("Inertia")

sns.lineplot(data=cluster_metrics, x="k", y="silhouette_score", marker="o", ax=axes[1])
axes[1].set_title("Silhouette Score")
axes[1].set_xlabel("Number of clusters")
axes[1].set_ylabel("Silhouette score")

plt.show()

### Insight

Use these metrics to choose a reasonable `k`. With a small dataset, the final choice should combine metrics with interpretability. A slightly lower metric can be acceptable if the clusters are easier to explain.

## Model Selection - Compare k=2 And k=3

### Question
Should we choose the statistically strongest model, or the model that creates more useful destination profiles?

### Method
The highest silhouette score is obtained for `k=2`, so this is the clearest separation statistically. However, because the dataset is small and this is a travel product, we also inspect `k=3` to see whether it creates more meaningful traveler-facing profiles.


In [ ]:
k2_assignments = pd.read_csv("../data/processed/kmeans_k2_assignments.csv")
k2_profiles = pd.read_csv("../data/processed/kmeans_k2_profiles.csv")

k3_assignments = pd.read_csv("../data/processed/kmeans_k3_assignments.csv")
k3_profiles = pd.read_csv("../data/processed/kmeans_k3_profiles.csv")

k2_profiles


In [ ]:
k2_assignments[["destination_name", "country", "cluster"]].sort_values(["cluster", "destination_name"])


In [ ]:
k3_profiles


In [ ]:
k3_assignments[["destination_name", "country", "cluster"]].sort_values(["cluster", "destination_name"])


### Insight
`k=2` separates large urban destinations from smaller/coastal/nature destinations. This is statistically strong, but quite broad.

`k=3` keeps the large urban destinations as one group, and then separates the remaining destinations into cooler balanced/nature destinations and warmer coastal/beach destinations. Even though its silhouette score is lower, `k=3` is more useful for explaining travel profiles.

For this version of the project, we continue with `k=3` as the main product-oriented model, while documenting that `k=2` was the strongest statistical option.


## Step 4 - Train K-Means

### Question

What groups does K-Means discover?

### Model

In [ ]:
# k=2 had the highest silhouette score, but k=3 creates more useful
# traveler-facing profiles: urban/culture, cool balanced/nature, warm coastal/beach.
BEST_K = 3

kmeans = KMeans(n_clusters=BEST_K, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(X_scaled)

df[["destination_name", "country", "cluster", "dominant_travel_style", "climate_category"]].sort_values("cluster")


### Insight

At this stage, cluster numbers are just labels. Cluster `0` is not automatically better than cluster `1`. We need to inspect the profile of each cluster.

## Step 5 - PCA Visualization

### Question

Can we visualize the clusters in two dimensions?

### Method

PCA projects the standardized feature space into two principal components. This does not train the clusters; it only helps us visualize them.

In [ ]:
pca = PCA(n_components=2, random_state=42)
components = pca.fit_transform(X_scaled)

df["PC1"] = components[:, 0]
df["PC2"] = components[:, 1]

explained_variance = pca.explained_variance_ratio_
explained_variance

In [ ]:
plt.figure(figsize=(10, 8))
sns.scatterplot(data=df, x="PC1", y="PC2", hue="cluster", style="dominant_travel_style", s=120)

for _, row in df.iterrows():
    plt.text(row["PC1"] + 0.03, row["PC2"] + 0.03, row["destination_name"], fontsize=9)

plt.axhline(0, color="gray", linewidth=0.8)
plt.axvline(0, color="gray", linewidth=0.8)
plt.title("K-Means Clusters Visualized With PCA")
plt.xlabel(f"PC1 ({explained_variance[0]:.1%} variance)")
plt.ylabel(f"PC2 ({explained_variance[1]:.1%} variance)")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.show()

### Insight

PCA helps us see whether clusters are visually separated. If many points overlap, it may mean the data does not have strong cluster structure or that the chosen features need adjustment.

## Step 6 - Business Interpretation

### Question

What does each cluster represent from a travel perspective?

### Analysis

In [ ]:
cluster_profile = df.groupby("cluster")[model_features + [
    "food_score",
    "nightlife_score",
    "culture_score",
    "nature_score",
    "beach_score",
    "overall_score",
]].mean().round(2)

cluster_profile

In [ ]:
for cluster_id in sorted(df["cluster"].unique()):
    print(f"Cluster {cluster_id}")
    display(df[df["cluster"] == cluster_id][[
        "destination_name",
        "country",
        "dominant_travel_style",
        "climate_category",
        "overall_score",
        "food_score",
        "nightlife_score",
        "culture_score",
        "nature_score",
        "beach_score",
    ]].sort_values("overall_score", ascending=False))
    print()

### Insight

Cluster interpretation should be based on both the numeric profile and the destination names inside each cluster. We should avoid saying only `Cluster 0`; instead, we try to describe it as a travel type such as Beach Destinations, City Breaks, Nature/Mountain, or Balanced/Mixed.

### Cluster Profile Names

After inspecting cluster profiles, we translate numeric cluster labels into traveler-facing profile names.

These names are intentionally editable. The model discovers groups, but the data scientist interprets what those groups mean for a traveler.

In [ ]:
cluster_profile_names = {
    0: "Cool Balanced & Nature",
    1: "Urban Culture & Food",
    2: "Warm Coastal & Beach",
}

df["cluster_profile"] = df["cluster"].map(cluster_profile_names)

df[["destination_name", "country", "cluster", "cluster_profile", "dominant_travel_style", "climate_category"]].sort_values("cluster")


## Step 7 - Generate Clustered Destination Profiles

### Question

How do we store the model output for the next sprint?

### Analysis

In [ ]:
OUTPUT_PATH = "../data/processed/destinations_clustered.csv"
df.to_csv(OUTPUT_PATH, index=False)
OUTPUT_PATH

### Insight

The clustered dataset can be used later by the recommendation engine. A user preference profile could first identify relevant clusters, then rank destinations inside those clusters.

## Final Notes

- K-Means is useful here because we want to discover natural groups without predefined labels.
- Standardization is mandatory because raw counts and weather values have different scales.
- PCA is used for visualization and interpretation, not as the recommendation engine itself.
- The next step is to inspect whether the discovered clusters are meaningful enough to support recommendations.